# Jira Onboarding Dataset Preparation

This notebook prepares the dataset used for the Intelligent Task Allocation System.

The base data is obtained from the public Apache Jira issue tracking dataset. Two Jira projects, AMBARI and IGNITE, were selected because they contain a large number of completed tasks distributed across multiple recurring assignees.

Only issues that:
- belong to AMBARI or IGNITE,
- have an assignee,
- have a status of Closed or Resolved, and
- belong to an agent with at least 20 historical assignments

are retained.

The resulting dataset contains approximately 36,000 real historical task assignments and 240 eligible agents.

The Jira data provides the real task-assignee history used as the foundation of the recommender system. Additional onboarding-specific attributes such as location, availability, workload, task type, skills and difficulty will later be derived or synthetically added to adapt the data to the distributed onboarding domain.

In [1]:
from pathlib import Path
import gzip
from bson import decode_file_iter
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd()

# If the notebook is running from inside the notebooks folder,
# move back to the main project directory.
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "jira_onboarding_v2" / "raw" / "issues.bson.gz"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "jira_onboarding_v2" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw Jira file:", RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())

Project root: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation
Raw Jira file: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\data\jira_onboarding_v2\raw\issues.bson.gz
File exists: True


## 1. Extract Relevant Jira Issues

We first extract completed issues from the AMBARI and IGNITE projects that have a recorded assignee. Only fields relevant to task allocation and later domain adaptation are retained.

In [2]:
SELECTED_PROJECTS = {"AMBARI", "IGNITE"}
COMPLETED_STATUSES = {"Closed", "Resolved"}

records = []

with gzip.open(RAW_DATA_PATH, "rb") as f:
    for issue in decode_file_iter(f):

        project = issue.get("projectname")
        assignee = issue.get("assignee")
        status = (issue.get("status") or {}).get("name")

        if (
            project in SELECTED_PROJECTS
            and assignee
            and status in COMPLETED_STATUSES
        ):
            components = [
                component.get("name")
                for component in (issue.get("components") or [])
                if component.get("name")
            ]

            records.append({
                "task_id": issue.get("key"),
                "project": project,
                "assignee": assignee,
                "issue_type": (issue.get("issuetype") or {}).get("name"),
                "components": components,
                "priority": (issue.get("priority") or {}).get("name"),
                "status": status,
                "created": issue.get("created"),
                "updated": issue.get("updated"),
                "summary": issue.get("summary"),
                "description": issue.get("description"),
                "labels": issue.get("labels") or []
            })

jira_df = pd.DataFrame(records)

print("Extracted records:", len(jira_df))
print("Unique assignees:", jira_df["assignee"].nunique())
print()
print(jira_df["project"].value_counts())

Extracted records: 38193
Unique assignees: 666

project
AMBARI    23049
IGNITE    15144
Name: count, dtype: int64


## 2. Filter Agents with Sufficient History

To ensure that each agent has enough historical information for recommendation, agents with fewer than 20 completed assignments are removed.

This reduces noise from very infrequent assignees and gives the models enough interaction history to learn meaningful patterns.

In [3]:
MIN_ASSIGNMENTS = 20

agent_counts = jira_df["assignee"].value_counts()

eligible_agents = agent_counts[
    agent_counts >= MIN_ASSIGNMENTS
].index

filtered_df = jira_df[
    jira_df["assignee"].isin(eligible_agents)
].copy()

print("Records before filtering:", len(jira_df))
print("Records after filtering:", len(filtered_df))
print("Agents before filtering:", jira_df["assignee"].nunique())
print("Agents after filtering:", filtered_df["assignee"].nunique())
print()
print(filtered_df["project"].value_counts())

Records before filtering: 38193
Records after filtering: 36387
Agents before filtering: 666
Agents after filtering: 240

project
AMBARI    22185
IGNITE    14202
Name: count, dtype: int64


## 3. Data Quality Checks

Before adapting the Jira data to the onboarding domain, the filtered dataset is checked for duplicate tasks, missing values and inconsistent fields.

This ensures that the synthetic and derived onboarding attributes are built on a clean real-data foundation.

In [4]:
# Check basic data quality

print("Dataset shape:", filtered_df.shape)
print("Unique tasks:", filtered_df["task_id"].nunique())
print("Duplicate task IDs:", filtered_df["task_id"].duplicated().sum())

print("\nMissing values:")
print(
    filtered_df[
        [
            "task_id",
            "project",
            "assignee",
            "issue_type",
            "priority",
            "created",
            "updated",
            "summary"
        ]
    ].isnull().sum()
)

print("\nIssue types:")
print(filtered_df["issue_type"].value_counts())

print("\nPriorities:")
print(filtered_df["priority"].value_counts())

print("\nTasks with no components:")
print((filtered_df["components"].str.len() == 0).sum())

Dataset shape: (36387, 12)
Unique tasks: 36387
Duplicate task IDs: 0

Missing values:
task_id       0
project       0
assignee      0
issue_type    0
priority      0
created       0
updated       0
summary       0
dtype: int64

Issue types:
issue_type
Bug               21385
Task               6925
Improvement        5301
Sub-task           1515
New Feature         679
Test                269
Technical task      108
Epic                 86
Story                82
Documentation        19
Wish                 18
Name: count, dtype: int64

Priorities:
priority
Major       26254
Critical     5922
Blocker      2308
Minor        1662
Trivial       241
Name: count, dtype: int64

Tasks with no components:
10391


## 4. Clean and Standardise the Base Data

The retained Jira records are cleaned and standardised before domain adaptation.

Tasks without a component are assigned a general component category rather than being removed, since the remaining task information is still useful. Text fields are also cleaned and timestamps are converted into a consistent format.

The original Jira assignee remains unchanged because it will be used as the historical ground-truth assignment.

In [5]:
clean_df = filtered_df.copy()

# Fill optional text fields
clean_df["summary"] = clean_df["summary"].fillna("").astype(str)
clean_df["description"] = clean_df["description"].fillna("").astype(str)

# Convert timestamps
clean_df["created"] = pd.to_datetime(clean_df["created"])
clean_df["updated"] = pd.to_datetime(clean_df["updated"])

# Ensure component lists are usable
clean_df["components"] = clean_df["components"].apply(
    lambda x: x if isinstance(x, list) and len(x) > 0 else ["General"]
)

# Use the first component as a simple primary component
clean_df["primary_component"] = clean_df["components"].apply(lambda x: x[0])

# Number of components associated with each task
clean_df["component_count"] = clean_df["components"].apply(len)

# Clean labels
clean_df["labels"] = clean_df["labels"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Sort chronologically
clean_df = clean_df.sort_values("created").reset_index(drop=True)

print("Clean dataset shape:", clean_df.shape)
print("Unique tasks:", clean_df["task_id"].nunique())
print("Unique agents:", clean_df["assignee"].nunique())
print("General component tasks:", (clean_df["primary_component"] == "General").sum())

print("\nDate range:")
print(clean_df["created"].min(), "to", clean_df["created"].max())

clean_df.head()

Clean dataset shape: (36387, 14)
Unique tasks: 36387
Unique agents: 240
General component tasks: 10391

Date range:
2011-09-22 16:11:31 to 2024-10-31 15:37:19


,task_id,project,assignee,issue_type,components,priority,status,created,updated,summary,description,labels,primary_component,component_count
0,AMBARI-1,AMBARI,owen.omalley,Task,[General],Major,Resolved,2011-09-22 16:11:31,2016-08-10 23:30:01,Initial code import,Initial code import,[],General,1
1,AMBARI-2,AMBARI,eyang,New Feature,[ambari-server],Major,Resolved,2011-09-22 22:57:33,2012-05-18 03:47:31,Define data model for agent controller communi...,Define the data structure for heartbeat from A...,[],ambari-server,1
2,AMBARI-3,AMBARI,eyang,Task,[infra],Major,Resolved,2011-09-22 23:06:14,2011-09-23 17:43:18,Move HMS Prototype to branch 0.0,HMS prototype is a working system. It is diff...,[],infra,1
3,AMBARI-4,AMBARI,owen.omalley,New Feature,[General],Major,Resolved,2011-09-22 23:17:57,2011-09-22 23:25:07,Create interface for component plugins,We should have a plugin interface for components.,[],General,1
4,AMBARI-5,AMBARI,vitthal_gogate,New Feature,[ambari-server],Major,Resolved,2011-09-22 23:30:33,2011-09-23 13:11:02,Added some left over changes from git reposito...,Adding some left over changes to svn after ini...,[],ambari-server,1


## 5. Chronological Train, Validation and Test Split

The dataset is divided chronologically rather than randomly.

The oldest 70% of tasks are used for training, the next 15% for validation, and the most recent 15% for final testing.

This better reflects the real task-allocation scenario, where past assignments are used to recommend agents for future tasks. The test set will remain untouched until the final model evaluation.

In [6]:
# Chronological 70/15/15 split

n = len(clean_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = clean_df.iloc[:train_end].copy()
validation_df = clean_df.iloc[train_end:validation_end].copy()
test_df = clean_df.iloc[validation_end:].copy()

print("Training records:", len(train_df))
print("Validation records:", len(validation_df))
print("Test records:", len(test_df))

print("\nTraining period:")
print(train_df["created"].min(), "to", train_df["created"].max())

print("\nValidation period:")
print(validation_df["created"].min(), "to", validation_df["created"].max())

print("\nTest period:")
print(test_df["created"].min(), "to", test_df["created"].max())

print("\nUnique agents:")
print("Train:", train_df["assignee"].nunique())
print("Validation:", validation_df["assignee"].nunique())
print("Test:", test_df["assignee"].nunique())

Training records: 25470
Validation records: 5458
Test records: 5459

Training period:
2011-09-22 16:11:31 to 2018-05-16 11:11:43

Validation period:
2018-05-16 11:30:56 to 2021-08-20 16:30:51

Test period:
2021-08-20 16:44:07 to 2024-10-31 15:37:19

Unique agents:
Train: 203
Validation: 145
Test: 69


## 6. Check for Cold-Start Agents

Before building agent profiles, the validation and test sets are checked for agents who do not appear in the training data.

Agents that have no training history cannot be meaningfully recommended by the collaborative filtering, content-based or knowledge graph components. This check helps ensure that the evaluation focuses on agents with sufficient historical information.

In [7]:
train_agents = set(train_df["assignee"])
validation_agents = set(validation_df["assignee"])
test_agents = set(test_df["assignee"])

unseen_validation = validation_agents - train_agents
unseen_test = test_agents - train_agents

print("Validation agents:", len(validation_agents))
print("Unseen validation agents:", len(unseen_validation))

print("\nTest agents:", len(test_agents))
print("Unseen test agents:", len(unseen_test))

print("\nValidation records with unseen agents:")
print(validation_df["assignee"].isin(unseen_validation).sum())

print("\nTest records with unseen agents:")
print(test_df["assignee"].isin(unseen_test).sum())

print("\nProject distribution:")
print("\nTRAIN")
print(train_df["project"].value_counts())

print("\nVALIDATION")
print(validation_df["project"].value_counts())

print("\nTEST")
print(test_df["project"].value_counts())

Validation agents: 145
Unseen validation agents: 27

Test agents: 69
Unseen test agents: 36

Validation records with unseen agents:
653

Test records with unseen agents:
3072

Project distribution:

TRAIN
project
AMBARI    20718
IGNITE     4752
Name: count, dtype: int64

VALIDATION
project
IGNITE    4126
AMBARI    1332
Name: count, dtype: int64

TEST
project
IGNITE    5324
AMBARI     135
Name: count, dtype: int64


## 7. Project-Aware Chronological Split

The initial global chronological split produced a large cold-start problem because the AMBARI and IGNITE projects cover different time periods.

To preserve the temporal nature of the evaluation while ensuring that both projects are represented throughout the dataset, each project is split chronologically into 70% training, 15% validation and 15% testing data.

The project-level splits are then combined.

In [8]:
train_parts = []
validation_parts = []
test_parts = []

for project, project_df in clean_df.groupby("project"):

    project_df = project_df.sort_values("created").reset_index(drop=True)

    n = len(project_df)

    train_end = int(n * 0.70)
    validation_end = int(n * 0.85)

    train_parts.append(project_df.iloc[:train_end])
    validation_parts.append(project_df.iloc[train_end:validation_end])
    test_parts.append(project_df.iloc[validation_end:])

train_df = pd.concat(train_parts).sort_values("created").reset_index(drop=True)
validation_df = pd.concat(validation_parts).sort_values("created").reset_index(drop=True)
test_df = pd.concat(test_parts).sort_values("created").reset_index(drop=True)

print("Training records:", len(train_df))
print("Validation records:", len(validation_df))
print("Test records:", len(test_df))

print("\nTRAIN")
print(train_df["project"].value_counts())

print("\nVALIDATION")
print(validation_df["project"].value_counts())

print("\nTEST")
print(test_df["project"].value_counts())

Training records: 25470
Validation records: 5458
Test records: 5459

TRAIN
project
AMBARI    15529
IGNITE     9941
Name: count, dtype: int64

VALIDATION
project
AMBARI    3328
IGNITE    2130
Name: count, dtype: int64

TEST
project
AMBARI    3328
IGNITE    2131
Name: count, dtype: int64


In [9]:
train_agents = set(train_df["assignee"])
validation_agents = set(validation_df["assignee"])
test_agents = set(test_df["assignee"])

unseen_validation = validation_agents - train_agents
unseen_test = test_agents - train_agents

print("Unseen validation agents:", len(unseen_validation))
print("Validation records with unseen agents:",
      validation_df["assignee"].isin(unseen_validation).sum())

print("\nUnseen test agents:", len(unseen_test))
print("Test records with unseen agents:",
      test_df["assignee"].isin(unseen_test).sum())

Unseen validation agents: 12
Validation records with unseen agents: 342

Unseen test agents: 19
Test records with unseen agents: 1015


## 8. Determine Warm-Start Agent Eligibility

The recommender models require historical information about an agent before they can make meaningful recommendations.

Therefore, different minimum training-history thresholds are compared. This helps select a reasonable threshold that provides sufficient agent history without removing too much validation and test data.

In [10]:
train_history = train_df["assignee"].value_counts()

for threshold in [1, 5, 10, 20]:

    eligible = set(
        train_history[train_history >= threshold].index
    )

    val_retained = validation_df[
        validation_df["assignee"].isin(eligible)
    ]

    test_retained = test_df[
        test_df["assignee"].isin(eligible)
    ]

    print(f"\nMinimum training assignments: {threshold}")
    print("Eligible agents:", len(eligible))

    print(
        "Validation records retained:",
        len(val_retained),
        f"({len(val_retained) / len(validation_df) * 100:.2f}%)"
    )

    print(
        "Test records retained:",
        len(test_retained),
        f"({len(test_retained) / len(test_df) * 100:.2f}%)"
    )


Minimum training assignments: 1
Eligible agents: 218
Validation records retained: 5116 (93.73%)
Test records retained: 4444 (81.41%)

Minimum training assignments: 5
Eligible agents: 204
Validation records retained: 4626 (84.76%)
Test records retained: 3996 (73.20%)

Minimum training assignments: 10
Eligible agents: 194
Validation records retained: 4349 (79.68%)
Test records retained: 3758 (68.84%)

Minimum training assignments: 20
Eligible agents: 187
Validation records retained: 4267 (78.18%)
Test records retained: 3751 (68.71%)


## 9. Create the Final Warm-Start Dataset

A minimum of 10 historical training assignments per agent is selected.

This threshold provides sufficient historical information for building agent profiles while retaining a reasonable amount of validation and test data.

Only agents that meet this threshold in the training data are included in the main recommender evaluation.

In [11]:
MIN_TRAIN_HISTORY = 10

train_history = train_df["assignee"].value_counts()

eligible_agents = set(
    train_history[
        train_history >= MIN_TRAIN_HISTORY
    ].index
)

final_train_df = train_df[
    train_df["assignee"].isin(eligible_agents)
].copy()

final_validation_df = validation_df[
    validation_df["assignee"].isin(eligible_agents)
].copy()

final_test_df = test_df[
    test_df["assignee"].isin(eligible_agents)
].copy()

print("Eligible agents:", len(eligible_agents))

print("\nFinal training records:", len(final_train_df))
print("Final validation records:", len(final_validation_df))
print("Final test records:", len(final_test_df))

print("\nProject distribution - Train")
print(final_train_df["project"].value_counts())

print("\nProject distribution - Validation")
print(final_validation_df["project"].value_counts())

print("\nProject distribution - Test")
print(final_test_df["project"].value_counts())

Eligible agents: 194

Final training records: 25370
Final validation records: 4349
Final test records: 3758

Project distribution - Train
project
AMBARI    15469
IGNITE     9901
Name: count, dtype: int64

Project distribution - Validation
project
AMBARI    2622
IGNITE    1727
Name: count, dtype: int64

Project distribution - Test
project
AMBARI    2134
IGNITE    1624
Name: count, dtype: int64


## 10. Create Clean Agent and Task Identifiers

The original Jira usernames are replaced with anonymous agent identifiers.

The original Jira task key is retained as a source reference, while a new task identifier is created for the onboarding dataset. This makes the prepared dataset easier to use within the Intelligent Task Allocation System while preserving traceability to the original public data.

In [12]:
# Create consistent anonymous agent IDs

sorted_agents = sorted(eligible_agents)

agent_id_map = {
    agent: f"AGENT_{i:03d}"
    for i, agent in enumerate(sorted_agents, start=1)
}

def add_clean_ids(df):
    df = df.copy()

    # Keep original Jira task ID for traceability
    df["source_task_id"] = df["task_id"]

    # Replace Jira username with anonymous agent ID
    df["agent_id"] = df["assignee"].map(agent_id_map)

    return df


final_train_df = add_clean_ids(final_train_df)
final_validation_df = add_clean_ids(final_validation_df)
final_test_df = add_clean_ids(final_test_df)

print("Mapped agents:", len(agent_id_map))

print("\nTraining sample:")
print(
    final_train_df[
        ["source_task_id", "agent_id", "project", "issue_type"]
    ].head()
)

Mapped agents: 194

Training sample:
  source_task_id   agent_id project   issue_type
0       AMBARI-1  AGENT_125  AMBARI         Task
1       AMBARI-2  AGENT_063  AMBARI  New Feature
2       AMBARI-3  AGENT_063  AMBARI         Task
3       AMBARI-4  AGENT_125  AMBARI  New Feature
4       AMBARI-5  AGENT_182  AMBARI  New Feature


## 11. Inspect Real Component Specialisation

Jira components provide useful information about the type of work associated with each task.

Before converting the Jira data into the onboarding domain, the most common components in the training data are examined. These components will later be mapped consistently to onboarding task categories.

Using the real component structure helps preserve genuine historical agent specialisation rather than assigning synthetic task types randomly.

In [13]:
component_counts = (
    final_train_df["primary_component"]
    .value_counts()
)

print("Unique training components:", len(component_counts))

print("\nTop 30 components:")
print(component_counts.head(30))

Unique training components: 68

Top 30 components:
primary_component
General             6776
ambari-web          6034
ambari-server       3676
ambari-agent         858
platforms            826
sql                  782
cache                698
general              595
wizards              489
documentation        476
stacks               383
ambari-metrics       345
ml                   318
ambari-views         305
persistence          229
contrib              201
odbc                 183
mvcc                 181
hadoop               180
igfs                 154
thin client          136
jdbc                 132
ambari-admin         124
build                107
binary                80
alerts                75
control.sh            74
ambari-logsearch      73
UI                    71
site                  59
Name: count, dtype: int64


## 12. Check Component Coverage

Before creating onboarding task categories, the component values in the validation and test sets are compared with those observed in training.

This ensures that the task-type mapping can be created from training data and applied consistently to future tasks without using information from the validation or test sets.

In [14]:
train_components = set(final_train_df["primary_component"])
validation_components = set(final_validation_df["primary_component"])
test_components = set(final_test_df["primary_component"])

unseen_validation_components = validation_components - train_components
unseen_test_components = test_components - train_components

print("Training components:", len(train_components))

print("\nValidation components:", len(validation_components))
print("Unseen validation components:", len(unseen_validation_components))
print(sorted(unseen_validation_components))

print("\nTest components:", len(test_components))
print("Unseen test components:", len(unseen_test_components))
print(sorted(unseen_test_components))

print("\nValidation records with unseen components:",
      final_validation_df["primary_component"]
      .isin(unseen_validation_components).sum())

print("Test records with unseen components:",
      final_test_df["primary_component"]
      .isin(unseen_test_components).sum())

Training components: 68

Validation components: 46
Unseen validation components: 8
['.NET', 'ambari-infra', 'ambari-sever', 'ambari-trunk', 'cli', 'express-upgrade', 'logsearch', 'service-advisor']

Test components: 41
Unseen test components: 7
['.NET', 'ambari-infra', 'ambari-server-spi', 'ambari-sever', 'cli', 'logsearch', 'service-advisor']

Validation records with unseen components: 40
Test records with unseen components: 103


## 13. Create Onboarding Task Types

The original Jira components are converted into onboarding task categories to adapt the public software-task dataset to the distributed onboarding domain.

The mapping is created using training data only. Each Jira component is consistently mapped to one onboarding task type so that historical agent specialisation is preserved.

Components that appear later in validation or testing but were not observed during training are assigned to a general onboarding category.

In [15]:
ONBOARDING_TASK_TYPES = [
    "KYC Verification",
    "Customer Registration",
    "SIM Registration",
    "M-PESA Onboarding",
    "Account Activation",
    "Document Verification"
]

# Count components using training data only
component_counts = final_train_df["primary_component"].value_counts()

# Balance components across the six task types
task_type_load = {task_type: 0 for task_type in ONBOARDING_TASK_TYPES}
component_to_task_type = {}

for component, count in component_counts.items():

    # Assign the component to the currently least-used task type
    selected_type = min(task_type_load, key=task_type_load.get)

    component_to_task_type[component] = selected_type
    task_type_load[selected_type] += count


def add_task_type(df):
    df = df.copy()

    df["task_type"] = (
        df["primary_component"]
        .map(component_to_task_type)
        .fillna("Customer Registration")
    )

    return df


final_train_df = add_task_type(final_train_df)
final_validation_df = add_task_type(final_validation_df)
final_test_df = add_task_type(final_test_df)

print("Training task-type distribution:")
print(final_train_df["task_type"].value_counts())

print("\nValidation task-type distribution:")
print(final_validation_df["task_type"].value_counts())

print("\nTest task-type distribution:")
print(final_test_df["task_type"].value_counts())

Training task-type distribution:
task_type
KYC Verification         6776
Customer Registration    6034
SIM Registration         3676
M-PESA Onboarding        2962
Document Verification    2961
Account Activation       2961
Name: count, dtype: int64

Validation task-type distribution:
task_type
KYC Verification         1442
SIM Registration          838
Customer Registration     660
Document Verification     618
M-PESA Onboarding         469
Account Activation        322
Name: count, dtype: int64

Test task-type distribution:
task_type
KYC Verification         1525
Customer Registration     735
SIM Registration          622
Document Verification     523
M-PESA Onboarding         222
Account Activation        131
Name: count, dtype: int64


## 14. Create Required Skills for Each Task Type

Each onboarding task type is associated with a small set of relevant skills.

These required skills are synthetically defined as part of the domain adaptation process. The mapping is fixed and applied consistently across training, validation and test data.

Agent skills will later be derived from their historical training assignments so that the original task-assignee patterns are preserved.

In [16]:
TASK_SKILLS = {
    "KYC Verification": [
        "Identity Verification",
        "Compliance",
        "Document Review"
    ],

    "Customer Registration": [
        "Customer Service",
        "Data Entry"
    ],

    "SIM Registration": [
        "Identity Verification",
        "SIM Registration",
        "Data Entry"
    ],

    "M-PESA Onboarding": [
        "Mobile Money",
        "Customer Service",
        "Compliance"
    ],

    "Account Activation": [
        "Account Setup",
        "Customer Service",
        "Data Entry"
    ],

    "Document Verification": [
        "Document Review",
        "Identity Verification",
        "Compliance"
    ]
}


def add_required_skills(df):
    df = df.copy()

    df["required_skills"] = df["task_type"].map(TASK_SKILLS)

    return df


final_train_df = add_required_skills(final_train_df)
final_validation_df = add_required_skills(final_validation_df)
final_test_df = add_required_skills(final_test_df)


ALL_SKILLS = sorted(
    set(
        skill
        for skills in TASK_SKILLS.values()
        for skill in skills
    )
)

print("Number of skills:", len(ALL_SKILLS))
print("\nSkills:")
for skill in ALL_SKILLS:
    print("-", skill)

print("\nSample tasks:")
print(
    final_train_df[
        ["source_task_id", "task_type", "required_skills"]
    ].head(10)
)

Number of skills: 8

Skills:
- Account Setup
- Compliance
- Customer Service
- Data Entry
- Document Review
- Identity Verification
- Mobile Money
- SIM Registration

Sample tasks:
  source_task_id              task_type  \
0       AMBARI-1       KYC Verification   
1       AMBARI-2       SIM Registration   
2       AMBARI-3  Document Verification   
3       AMBARI-4       KYC Verification   
4       AMBARI-5       SIM Registration   
5       AMBARI-6       KYC Verification   
6       AMBARI-7       SIM Registration   
7       AMBARI-8       KYC Verification   
8       AMBARI-9  Document Verification   
9      AMBARI-10       KYC Verification   

                                     required_skills  
0  [Identity Verification, Compliance, Document R...  
1  [Identity Verification, SIM Registration, Data...  
2  [Document Review, Identity Verification, Compl...  
3  [Identity Verification, Compliance, Document R...  
4  [Identity Verification, SIM Registration, Data...  
5  [Identity Ve

## 15. Derive Agent Skills from Historical Assignments

Agent skills are derived only from the training data.

For each agent, the system counts how often they completed tasks requiring each onboarding skill. A skill proficiency score is then calculated as the proportion of the agent's historical training tasks that required that skill.

This preserves the real task-assignment patterns while creating interpretable onboarding skill profiles without using validation or test information.

In [17]:
# Number of historical training assignments per agent
agent_task_counts = (
    final_train_df["agent_id"]
    .value_counts()
    .rename("historical_task_count")
)

# Expand each task into one row per required skill
skill_history = (
    final_train_df[
        ["agent_id", "source_task_id", "required_skills"]
    ]
    .explode("required_skills")
)

# Count how often each agent handled each skill
agent_skill_counts = (
    skill_history
    .groupby(["agent_id", "required_skills"])
    .size()
    .reset_index(name="skill_task_count")
)

# Add each agent's total historical task count
agent_skill_counts["historical_task_count"] = (
    agent_skill_counts["agent_id"]
    .map(agent_task_counts)
)

# Proportion of the agent's historical tasks involving the skill
agent_skill_counts["skill_proficiency"] = (
    agent_skill_counts["skill_task_count"]
    / agent_skill_counts["historical_task_count"]
)

# Create a wide proficiency table
agent_skill_matrix = (
    agent_skill_counts
    .pivot(
        index="agent_id",
        columns="required_skills",
        values="skill_proficiency"
    )
    .fillna(0)
)

# Make sure all 8 skills exist as columns
agent_skill_matrix = agent_skill_matrix.reindex(
    columns=ALL_SKILLS,
    fill_value=0
)

print("Agents with skill profiles:", len(agent_skill_matrix))
print("Skills:", len(agent_skill_matrix.columns))

agent_skill_matrix.head()

Agents with skill profiles: 194
Skills: 8


required_skills,Account Setup,Compliance,Customer Service,Data Entry,Document Review,Identity Verification,Mobile Money,SIM Registration
agent_id,,,,,,,,
AGENT_001,0.909091,0.090909,0.909091,0.909091,0.090909,0.090909,0.000000,0.000000
AGENT_002,0.043478,0.956522,0.391304,0.043478,0.608696,0.608696,0.347826,0.000000
AGENT_003,0.000000,0.047414,0.984914,0.952586,0.006466,0.015086,0.040948,0.008621
AGENT_004,0.000000,0.045220,0.963824,0.954780,0.010336,0.036176,0.034884,0.025840
AGENT_005,0.000000,0.976744,0.023256,0.023256,0.953488,0.976744,0.023256,0.023256


## 16. Identify Each Agent's Strongest Skills

Each agent's strongest onboarding skills are selected from the skill proficiency scores derived from their training history.

The top three skills are retained as the agent's main skill profile. This keeps the recommender simple and interpretable while still reflecting the agent's real historical task patterns.

In [18]:
# Select the top 3 skills for each agent

def get_top_skills(row, top_n=3):
    return (
        row.sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )


agent_top_skills = agent_skill_matrix.apply(
    get_top_skills,
    axis=1
)

agent_profiles = pd.DataFrame({
    "agent_id": agent_skill_matrix.index,
    "top_skills": agent_top_skills.values
})

# Add historical task count
agent_profiles["historical_task_count"] = (
    agent_profiles["agent_id"]
    .map(agent_task_counts)
)

print("Agent profiles created:", len(agent_profiles))

agent_profiles.head(10)

Agent profiles created: 194


,agent_id,top_skills,historical_task_count
0,AGENT_001,"[Account Setup, Customer Service, Data Entry]",11
1,AGENT_002,"[Compliance, Document Review, Identity Verific...",23
2,AGENT_003,"[Customer Service, Data Entry, Compliance]",928
3,AGENT_004,"[Customer Service, Data Entry, Compliance]",774
4,AGENT_005,"[Compliance, Identity Verification, Document R...",43
5,AGENT_006,"[Identity Verification, Data Entry, SIM Regist...",122
6,AGENT_007,"[Identity Verification, Compliance, Document R...",21
7,AGENT_008,"[Data Entry, Identity Verification, SIM Regist...",247
8,AGENT_009,"[Compliance, Document Review, Identity Verific...",274
9,AGENT_010,"[Compliance, Document Review, Identity Verific...",118


## 17. Derive Agent Task-Type Experience

Agent task-type experience is calculated from the training data only.

For each agent, the system counts how many historical tasks they completed in each onboarding task category. The most frequently handled task types are then used to represent the agent's area of experience.

This feature provides a simple and interpretable measure of domain familiarity for the recommender.

In [19]:
# Count historical task types handled by each agent
agent_task_type_counts = (
    final_train_df
    .groupby(["agent_id", "task_type"])
    .size()
    .reset_index(name="task_type_count")
)

# Create task-type experience matrix
agent_task_type_matrix = (
    agent_task_type_counts
    .pivot(
        index="agent_id",
        columns="task_type",
        values="task_type_count"
    )
    .fillna(0)
)

# Make sure all task types are present
agent_task_type_matrix = agent_task_type_matrix.reindex(
    columns=ONBOARDING_TASK_TYPES,
    fill_value=0
)

# Find top 2 most experienced task types for each agent
agent_top_task_types = agent_task_type_matrix.apply(
    lambda row: row.sort_values(ascending=False)
                   .head(2)
                   .index
                   .tolist(),
    axis=1
)

# Add to agent profiles
agent_profiles["experienced_task_types"] = (
    agent_profiles["agent_id"]
    .map(agent_top_task_types)
)

print("Agent profiles:", len(agent_profiles))

agent_profiles[
    [
        "agent_id",
        "top_skills",
        "experienced_task_types",
        "historical_task_count"
    ]
].head(10)

Agent profiles: 194


,agent_id,top_skills,experienced_task_types,historical_task_count
0,AGENT_001,"[Account Setup, Customer Service, Data Entry]","[Account Activation, KYC Verification]",11
1,AGENT_002,"[Compliance, Document Review, Identity Verific...","[KYC Verification, M-PESA Onboarding]",23
2,AGENT_003,"[Customer Service, Data Entry, Compliance]","[Customer Registration, M-PESA Onboarding]",928
3,AGENT_004,"[Customer Service, Data Entry, Compliance]","[Customer Registration, M-PESA Onboarding]",774
4,AGENT_005,"[Compliance, Identity Verification, Document R...","[KYC Verification, SIM Registration]",43
5,AGENT_006,"[Identity Verification, Data Entry, SIM Regist...","[SIM Registration, Document Verification]",122
6,AGENT_007,"[Identity Verification, Compliance, Document R...","[KYC Verification, SIM Registration]",21
7,AGENT_008,"[Data Entry, Identity Verification, SIM Regist...","[SIM Registration, Account Activation]",247
8,AGENT_009,"[Compliance, Document Review, Identity Verific...","[KYC Verification, Document Verification]",274
9,AGENT_010,"[Compliance, Document Review, Identity Verific...","[KYC Verification, Document Verification]",118


## 18. Add Synthetic Operational Locations

The original Jira dataset does not contain field-agent locations.

To adapt the data to distributed onboarding operations, Kenyan locations are synthetically assigned using fixed and reproducible rules.

Agent locations are derived from their historical training specialisation, while task locations are derived from the onboarding task category. The same rules are applied to training, validation and test data.

This creates a consistent operational location feature without using future assignment information.

In [20]:
KENYAN_LOCATIONS = [
    "Nairobi",
    "Mombasa",
    "Kisumu",
    "Nakuru",
    "Eldoret",
    "Thika"
]

# Map onboarding task types to typical operating locations
TASK_TYPE_LOCATION = {
    "KYC Verification": "Nairobi",
    "Customer Registration": "Mombasa",
    "SIM Registration": "Kisumu",
    "M-PESA Onboarding": "Nakuru",
    "Account Activation": "Eldoret",
    "Document Verification": "Thika"
}

# Use each agent's most experienced task type
agent_primary_task_type = (
    agent_profiles["experienced_task_types"]
    .apply(lambda x: x[0])
)

agent_profiles["location"] = (
    agent_primary_task_type
    .map(TASK_TYPE_LOCATION)
)

def add_task_location(df):
    df = df.copy()

    df["task_location"] = (
        df["task_type"]
        .map(TASK_TYPE_LOCATION)
    )

    return df


final_train_df = add_task_location(final_train_df)
final_validation_df = add_task_location(final_validation_df)
final_test_df = add_task_location(final_test_df)

print("Agent location distribution:")
print(agent_profiles["location"].value_counts())

print("\nTraining task location distribution:")
print(final_train_df["task_location"].value_counts())

Agent location distribution:
location
Nairobi    78
Kisumu     30
Eldoret    23
Nakuru     23
Thika      21
Mombasa    19
Name: count, dtype: int64

Training task location distribution:
task_location
Nairobi    6776
Mombasa    6034
Kisumu     3676
Nakuru     2962
Thika      2961
Eldoret    2961
Name: count, dtype: int64


## 19. Derive Agent Workload and Availability

Operational workload is estimated from recent training activity.

For each agent, the number of tasks handled during the most recent period of the training data is used as a workload proxy. This value is converted into a simple workload level from 0 to 5.

Availability is then derived from workload, where agents with very high workload are considered unavailable for additional tasks.

These fields are derived only from training history and do not use validation or test assignments.

In [21]:
# Use the most recent 90 days of training data
latest_train_date = final_train_df["created"].max()
recent_start_date = latest_train_date - pd.Timedelta(days=90)

recent_train = final_train_df[
    final_train_df["created"] >= recent_start_date
]

# Count recent tasks per agent
recent_workload = (
    recent_train["agent_id"]
    .value_counts()
    .reindex(agent_profiles["agent_id"])
    .fillna(0)
)

# Convert recent activity into a simple workload score from 0 to 5
if recent_workload.max() > 0:
    workload_score = np.ceil(
        (recent_workload / recent_workload.max()) * 5
    ).astype(int)
else:
    workload_score = pd.Series(
        0,
        index=agent_profiles["agent_id"]
    )

workload_score = workload_score.clip(0, 5)

agent_profiles["current_workload"] = (
    agent_profiles["agent_id"]
    .map(workload_score)
)

# Availability rule
agent_profiles["availability"] = np.where(
    agent_profiles["current_workload"] >= 5,
    "Unavailable",
    "Available"
)

print("Training reference date:", latest_train_date)
print("Recent workload period:", recent_start_date, "to", latest_train_date)

print("\nWorkload distribution:")
print(
    agent_profiles["current_workload"]
    .value_counts()
    .sort_index()
)

print("\nAvailability:")
print(agent_profiles["availability"].value_counts())

agent_profiles[
    [
        "agent_id",
        "historical_task_count",
        "current_workload",
        "availability"
    ]
].head(10)

Training reference date: 2022-05-12 10:07:02
Recent workload period: 2022-02-11 10:07:02 to 2022-05-12 10:07:02

Workload distribution:
current_workload
0    161
1     16
2      6
3      6
4      3
5      2
Name: count, dtype: int64

Availability:
availability
Available      192
Unavailable      2
Name: count, dtype: int64


,agent_id,historical_task_count,current_workload,availability
0,AGENT_001,11,1,Available
1,AGENT_002,23,0,Available
2,AGENT_003,928,0,Available
3,AGENT_004,774,0,Available
4,AGENT_005,43,0,Available
5,AGENT_006,122,0,Available
6,AGENT_007,21,0,Available
7,AGENT_008,247,0,Available
8,AGENT_009,274,0,Available
9,AGENT_010,118,0,Available


## 20. Create Synthetic Operational Workload and Availability

The Jira dataset does not provide current field-agent workload or availability.

Therefore, these operational attributes are synthetically generated using fixed and reproducible rules. Workload is represented on a scale from 0 to 5, where higher values indicate a busier agent.

Availability is derived from workload, with agents at the highest workload level treated as unavailable.

These attributes are used as supporting operational features and are not intended to dominate the recommendation score.

In [22]:
# Reproducible synthetic operational state
rng = np.random.default_rng(42)

# Realistic workload distribution:
# most agents have low/moderate workload,
# while fewer agents are heavily loaded.
workload_levels = [0, 1, 2, 3, 4, 5]
workload_probabilities = [0.10, 0.20, 0.25, 0.20, 0.15, 0.10]

agent_profiles["current_workload"] = rng.choice(
    workload_levels,
    size=len(agent_profiles),
    p=workload_probabilities
)

agent_profiles["availability"] = np.where(
    agent_profiles["current_workload"] == 5,
    "Unavailable",
    "Available"
)

print("Workload distribution:")
print(
    agent_profiles["current_workload"]
    .value_counts()
    .sort_index()
)

print("\nAvailability distribution:")
print(agent_profiles["availability"].value_counts())

Workload distribution:
current_workload
0    18
1    39
2    47
3    42
4    32
5    16
Name: count, dtype: int64

Availability distribution:
availability
Available      178
Unavailable     16
Name: count, dtype: int64


## 21. Create Agent Experience and Performance

Agent experience is derived from the number of historical training assignments completed by each agent.

Agents are grouped into three simple experience levels: Junior, Intermediate and Senior.

The Jira dataset does not provide a direct field-agent performance score. Therefore, a reproducible synthetic performance score is generated as an operational attribute. The score is kept within a realistic range and will be used as a supporting feature rather than as the main recommendation signal.

In [23]:
# -------------------------
# Experience level
# -------------------------

def get_experience_level(task_count):
    if task_count < 50:
        return "Junior"
    elif task_count < 200:
        return "Intermediate"
    else:
        return "Senior"


agent_profiles["experience_level"] = (
    agent_profiles["historical_task_count"]
    .apply(get_experience_level)
)


# -------------------------
# Synthetic performance
# -------------------------

# Reproducible random generator
performance_rng = np.random.default_rng(42)

# Generate realistic performance scores between 0.70 and 0.98
agent_profiles["performance_score"] = np.round(
    performance_rng.uniform(
        0.70,
        0.98,
        size=len(agent_profiles)
    ),
    3
)


print("Experience distribution:")
print(agent_profiles["experience_level"].value_counts())

print("\nPerformance summary:")
print(agent_profiles["performance_score"].describe())

print("\nSample agent profiles:")
print(
    agent_profiles[
        [
            "agent_id",
            "historical_task_count",
            "experience_level",
            "performance_score",
            "current_workload",
            "availability"
        ]
    ].head(10)
)

Experience distribution:
experience_level
Junior          87
Intermediate    75
Senior          32
Name: count, dtype: int64

Performance summary:
count    194.000000
mean       0.839149
std        0.079000
min        0.702000
25%        0.773250
50%        0.835500
75%        0.909750
max        0.978000
Name: performance_score, dtype: float64

Sample agent profiles:
    agent_id  historical_task_count experience_level  performance_score  \
0  AGENT_001                     11           Junior              0.917   
1  AGENT_002                     23           Junior              0.823   
2  AGENT_003                    928           Senior              0.940   
3  AGENT_004                    774           Senior              0.895   
4  AGENT_005                     43           Junior              0.726   
5  AGENT_006                    122     Intermediate              0.973   
6  AGENT_007                     21           Junior              0.913   
7  AGENT_008                 

## 22. Derive Task Difficulty from Priority

Task difficulty is derived from the original Jira priority field.

Lower-priority tasks are treated as lower difficulty, while Critical and Blocker tasks are treated as high difficulty. This provides a simple task-complexity feature without introducing additional random data.

In [24]:
PRIORITY_TO_DIFFICULTY = {
    "Trivial": "Low",
    "Minor": "Low",
    "Major": "Medium",
    "Critical": "High",
    "Blocker": "High"
}


def add_task_difficulty(df):
    df = df.copy()

    df["difficulty"] = (
        df["priority"]
        .map(PRIORITY_TO_DIFFICULTY)
        .fillna("Medium")
    )

    return df


final_train_df = add_task_difficulty(final_train_df)
final_validation_df = add_task_difficulty(final_validation_df)
final_test_df = add_task_difficulty(final_test_df)


print("Training difficulty distribution:")
print(final_train_df["difficulty"].value_counts())

print("\nValidation difficulty distribution:")
print(final_validation_df["difficulty"].value_counts())

print("\nTest difficulty distribution:")
print(final_test_df["difficulty"].value_counts())

Training difficulty distribution:
difficulty
Medium    18636
High       5444
Low        1290
Name: count, dtype: int64

Validation difficulty distribution:
difficulty
Medium    2968
High      1186
Low        195
Name: count, dtype: int64

Test difficulty distribution:
difficulty
Medium    2537
High      1106
Low        115
Name: count, dtype: int64


## 23. Final Prepared Dataset Quality Check

Before saving the prepared datasets, final checks are performed to confirm that:

- task identifiers are unique,
- training, validation and test sets do not overlap,
- all evaluated agents exist in the training agent set,
- required task attributes are complete,
- agent profiles are complete, and
- the prepared dataset contains the expected number of agents and tasks.

After these checks, the prepared dataset structure is considered fixed for model development.

In [25]:
# --------------------------------------------------
# Final dataset quality checks
# --------------------------------------------------

print("=== DATASET SIZES ===")
print("Train:", len(final_train_df))
print("Validation:", len(final_validation_df))
print("Test:", len(final_test_df))
print("Agents:", len(agent_profiles))


# --------------------------------------------------
# Check task ID uniqueness
# --------------------------------------------------

print("\n=== DUPLICATE TASK IDS ===")
print("Train:", final_train_df["source_task_id"].duplicated().sum())
print("Validation:", final_validation_df["source_task_id"].duplicated().sum())
print("Test:", final_test_df["source_task_id"].duplicated().sum())


# --------------------------------------------------
# Check split overlap
# --------------------------------------------------

train_tasks = set(final_train_df["source_task_id"])
validation_tasks = set(final_validation_df["source_task_id"])
test_tasks = set(final_test_df["source_task_id"])

print("\n=== SPLIT OVERLAP ===")
print("Train / Validation:", len(train_tasks & validation_tasks))
print("Train / Test:", len(train_tasks & test_tasks))
print("Validation / Test:", len(validation_tasks & test_tasks))


# --------------------------------------------------
# Check agent coverage
# --------------------------------------------------

profile_agents = set(agent_profiles["agent_id"])

print("\n=== AGENT COVERAGE ===")
print(
    "Unknown agents in validation:",
    len(set(final_validation_df["agent_id"]) - profile_agents)
)

print(
    "Unknown agents in test:",
    len(set(final_test_df["agent_id"]) - profile_agents)
)


# --------------------------------------------------
# Check important task fields
# --------------------------------------------------

task_fields = [
    "source_task_id",
    "agent_id",
    "project",
    "task_type",
    "required_skills",
    "task_location",
    "difficulty",
    "created"
]

print("\n=== MISSING TASK VALUES ===")
print(final_train_df[task_fields].isnull().sum())


# --------------------------------------------------
# Check agent profile fields
# --------------------------------------------------

agent_fields = [
    "agent_id",
    "top_skills",
    "experienced_task_types",
    "historical_task_count",
    "location",
    "current_workload",
    "availability",
    "experience_level",
    "performance_score"
]

print("\n=== MISSING AGENT VALUES ===")
print(agent_profiles[agent_fields].isnull().sum())

=== DATASET SIZES ===
Train: 25370
Validation: 4349
Test: 3758
Agents: 194

=== DUPLICATE TASK IDS ===
Train: 0
Validation: 0
Test: 0

=== SPLIT OVERLAP ===
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

=== AGENT COVERAGE ===
Unknown agents in validation: 0
Unknown agents in test: 0

=== MISSING TASK VALUES ===
source_task_id     0
agent_id           0
project            0
task_type          0
required_skills    0
task_location      0
difficulty         0
created            0
dtype: int64

=== MISSING AGENT VALUES ===
agent_id                  0
top_skills                0
experienced_task_types    0
historical_task_count     0
location                  0
current_workload          0
availability              0
experience_level          0
performance_score         0
dtype: int64


## 24. Save the Prepared Dataset

The cleaned and domain-adapted dataset is saved into separate files for model development.

The saved files include:

- agent profiles,
- agent skill proficiency,
- training tasks,
- validation tasks,
- test tasks.

The training, validation and test splits are kept separate so that the test set remains untouched until final evaluation.

In [26]:
# --------------------------------------------------
# Prepare datasets for saving
# --------------------------------------------------

def prepare_task_file(df):
    output = df[
        [
            "source_task_id",
            "agent_id",
            "project",
            "task_type",
            "required_skills",
            "task_location",
            "difficulty",
            "priority",
            "issue_type",
            "primary_component",
            "created"
        ]
    ].copy()

    # Convert list columns into simple CSV-friendly text
    output["required_skills"] = output["required_skills"].apply(
        lambda x: "|".join(x)
    )

    return output


train_output = prepare_task_file(final_train_df)
validation_output = prepare_task_file(final_validation_df)
test_output = prepare_task_file(final_test_df)


# --------------------------------------------------
# Prepare agent profiles
# --------------------------------------------------

agents_output = agent_profiles.copy()

agents_output["top_skills"] = agents_output["top_skills"].apply(
    lambda x: "|".join(x)
)

agents_output["experienced_task_types"] = (
    agents_output["experienced_task_types"]
    .apply(lambda x: "|".join(x))
)


# --------------------------------------------------
# Prepare agent skill table
# --------------------------------------------------

agent_skills_output = (
    agent_skill_matrix
    .reset_index()
    .melt(
        id_vars="agent_id",
        var_name="skill",
        value_name="proficiency"
    )
)

agent_skills_output["proficiency"] = (
    agent_skills_output["proficiency"]
    .round(4)
)


# --------------------------------------------------
# Save files
# --------------------------------------------------

agents_output.to_csv(
    PROCESSED_DATA_DIR / "agents.csv",
    index=False
)

agent_skills_output.to_csv(
    PROCESSED_DATA_DIR / "agent_skills.csv",
    index=False
)

train_output.to_csv(
    PROCESSED_DATA_DIR / "train_tasks.csv",
    index=False
)

validation_output.to_csv(
    PROCESSED_DATA_DIR / "validation_tasks.csv",
    index=False
)

test_output.to_csv(
    PROCESSED_DATA_DIR / "test_tasks.csv",
    index=False
)


print("Saved prepared datasets:")
print("agents.csv:", len(agents_output))
print("agent_skills.csv:", len(agent_skills_output))
print("train_tasks.csv:", len(train_output))
print("validation_tasks.csv:", len(validation_output))
print("test_tasks.csv:", len(test_output))

Saved prepared datasets:
agents.csv: 194
agent_skills.csv: 1552
train_tasks.csv: 25370
validation_tasks.csv: 4349
test_tasks.csv: 3758


## 25. Verify Saved Dataset Files

The saved CSV files are loaded again to confirm that they were written correctly and can be used by the recommendation models in later notebooks.

In [27]:
# Reload saved files for verification

saved_agents = pd.read_csv(
    PROCESSED_DATA_DIR / "agents.csv"
)

saved_agent_skills = pd.read_csv(
    PROCESSED_DATA_DIR / "agent_skills.csv"
)

saved_train = pd.read_csv(
    PROCESSED_DATA_DIR / "train_tasks.csv"
)

saved_validation = pd.read_csv(
    PROCESSED_DATA_DIR / "validation_tasks.csv"
)

saved_test = pd.read_csv(
    PROCESSED_DATA_DIR / "test_tasks.csv"
)


print("agents.csv:", saved_agents.shape)
print("agent_skills.csv:", saved_agent_skills.shape)
print("train_tasks.csv:", saved_train.shape)
print("validation_tasks.csv:", saved_validation.shape)
print("test_tasks.csv:", saved_test.shape)

print("\nAgent columns:")
print(saved_agents.columns.tolist())

print("\nTask columns:")
print(saved_train.columns.tolist())

print("\nSample agent:")
display(saved_agents.head(3))

print("\nSample training task:")
display(saved_train.head(3))

agents.csv: (194, 9)
agent_skills.csv: (1552, 3)
train_tasks.csv: (25370, 11)
validation_tasks.csv: (4349, 11)
test_tasks.csv: (3758, 11)

Agent columns:
['agent_id', 'top_skills', 'historical_task_count', 'experienced_task_types', 'location', 'current_workload', 'availability', 'experience_level', 'performance_score']

Task columns:
['source_task_id', 'agent_id', 'project', 'task_type', 'required_skills', 'task_location', 'difficulty', 'priority', 'issue_type', 'primary_component', 'created']

Sample agent:


,agent_id,top_skills,historical_task_count,experienced_task_types,location,current_workload,availability,experience_level,performance_score
0,AGENT_001,Account Setup|Customer Service|Data Entry,11,Account Activation|KYC Verification,Eldoret,4,Available,Junior,0.917
1,AGENT_002,Compliance|Document Review|Identity Verification,23,KYC Verification|M-PESA Onboarding,Nairobi,2,Available,Junior,0.823
2,AGENT_003,Customer Service|Data Entry|Compliance,928,Customer Registration|M-PESA Onboarding,Mombasa,4,Available,Senior,0.940



Sample training task:


,source_task_id,agent_id,project,task_type,required_skills,task_location,difficulty,priority,issue_type,primary_component,created
0,AMBARI-1,AGENT_125,AMBARI,KYC Verification,Identity Verification|Compliance|Document Review,Nairobi,Medium,Major,Task,General,2011-09-22 16:11:31
1,AMBARI-2,AGENT_063,AMBARI,SIM Registration,Identity Verification|SIM Registration|Data Entry,Kisumu,Medium,Major,New Feature,ambari-server,2011-09-22 22:57:33
2,AMBARI-3,AGENT_063,AMBARI,Document Verification,Document Review|Identity Verification|Compliance,Thika,Medium,Major,Task,infra,2011-09-22 23:06:14


## 26. Data Preparation Summary

The final dataset was prepared from real historical Jira issue assignments from the AMBARI and IGNITE projects and adapted to the distributed onboarding domain.

### Final Dataset

- **194 eligible agents**
- **25,370 training tasks**
- **4,349 validation tasks**
- **3,758 test tasks**
- **6 onboarding task types**
- **8 onboarding skills**
- No duplicate task IDs
- No overlap between training, validation and test sets
- No unknown agents in the final validation or test sets
- No missing values in the required modelling fields

### Real / Derived Attributes

The following information originates from or is derived from the real Jira dataset:

- Historical task assignments
- Historical assignees
- Task timestamps
- Jira projects
- Components
- Issue types
- Priorities
- Agent task history
- Agent skill proficiency
- Agent task-type experience
- Experience level
- Task difficulty

### Synthetic Domain-Adaptation Attributes

The following attributes were added to adapt the dataset to distributed onboarding operations:

- Onboarding task-type labels
- Required onboarding skills
- Kenyan agent and task locations
- Current workload
- Availability
- Performance score

Synthetic attributes were generated using fixed and reproducible rules. The real historical Jira task-assignee relationship remains the ground truth used for recommender evaluation.

The prepared dataset is now ready for the development of the Content-Based Filtering, Collaborative Filtering, Knowledge Graph and Hybrid recommendation models.